<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

This notebook contains the executable calibration workflow. Problem definition, mathematical theory, and design rationale are maintained in the companion notebooks.

## Setup — Environment and Configuration

Imports, calibration constants, and the output directory used by all 13 required tasks.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Keep numerical output compact while retaining useful precision.
np.set_printoptions(precision=6, suppress=True)

In [ ]:
SQUARE_SIZE_M = 0.03
INTERNAL_CORNERS_X = 8
INTERNAL_CORNERS_Y = 6
MIN_VALID_VIEWS = 3

In [ ]:
# Directory used to store generated figures.
OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the Sorted JPEG Calibration Images

Discover the repository-relative calibration folder, load the `.jpg` filenames in deterministic sorted order, and fail explicitly if the input directory or images are missing.

In [ ]:
DATA_DIR = Path("../data/calibration_images")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Calibration image directory not found: {DATA_DIR}"
    )

image_paths = sorted(DATA_DIR.glob("*.jpg"))

if not image_paths:
    raise FileNotFoundError(
        f"No .jpg calibration images found in: {DATA_DIR}"
    )

loaded_views = []

for image_path in image_paths:
    image = cv2.imread(str(image_path))

    if image is None:
        print(
            f"[SKIPPED] {image_path.name}: "
            "image could not be read"
        )
        continue

    loaded_views.append(
        {
            "path": image_path,
            "name": image_path.name,
            "image": image,
        }
    )

if not loaded_views:
    raise RuntimeError(
        "No readable calibration images are available."
    )

print(f"Readable calibration images: {len(loaded_views)}")
for view in loaded_views:
    print(f"  - {view['name']}")

## 2. Detect and Refine Chessboard Corners

Use `detect_and_refine_corners()` with `cv2.findChessboardCorners`, then refine successful detections to sub-pixel precision with `cv2.cornerSubPix`.

In [ ]:
REFINE_CRITERIA = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    30,
    0.001,
)

def detect_and_refine_corners(
    image,
    corners_x=INTERNAL_CORNERS_X,
    corners_y=INTERNAL_CORNERS_Y,
):
    """Detect and refine the internal chessboard corners."""
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY,
    )

    found, corners = cv2.findChessboardCorners(
        gray,
        (corners_x, corners_y),
        None,
    )

    if not found or corners is None:
        return None

    refined = cv2.cornerSubPix(
        gray,
        corners,
        (11, 11),
        (-1, -1),
        REFINE_CRITERIA,
    )

    return refined.reshape(-1, 2)

views = []

for loaded_view in loaded_views:
    image_points = detect_and_refine_corners(
        loaded_view["image"]
    )

    if image_points is None:
        print(
            f"[SKIPPED] {loaded_view['name']}: "
            "chessboard not detected"
        )
        continue

    view = loaded_view.copy()
    view["im_pts"] = image_points
    views.append(view)

    print(
        f"[OK] {view['name']}: "
        f"{image_points.shape[0]} refined corners"
    )

if len(views) < MIN_VALID_VIEWS:
    raise RuntimeError(
        f"At least {MIN_VALID_VIEWS} valid calibration views "
        f"are required; only {len(views)} were detected."
    )

print(f"\nValid calibration views: {len(views)}")

## 3. Build the Planar World Coordinates

Use `build_planar_points()` to generate the $8 \times 6$ internal-corner grid with $0.03\,\mathrm{m}$ spacing and the first corner at the planar origin.

In [ ]:
def build_planar_points(
    corners_x=INTERNAL_CORNERS_X,
    corners_y=INTERNAL_CORNERS_Y,
    square_size=SQUARE_SIZE_M,
):
    """Return ordered planar chessboard coordinates in metres."""
    points = np.zeros(
        (corners_x * corners_y, 2),
        dtype=np.float64,
    )

    index = 0

    for y_index in range(corners_y):
        for x_index in range(corners_x):
            points[index, 0] = x_index * square_size
            points[index, 1] = y_index * square_size
            index += 1

    return points

plane_points = build_planar_points()

for view in views:
    view["plane_pts"] = plane_points.copy()

print(f"Planar points per view: {plane_points.shape[0]}")
print(f"Square size: {SQUARE_SIZE_M:.3f} m")

## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

`normalize_trans()` translates each 2D point set to zero centroid and scales its mean distance to $\sqrt{2}$.

In [ ]:
def homogenize(points):
    """Convert 2D Cartesian points to homogeneous coordinates."""
    points = np.asarray(points, dtype=float)

    homogeneous = np.ones((points.shape[0], 3), dtype=float)
    homogeneous[:, :2] = points

    return homogeneous

def normalize_trans(points):
    """Return the 2D similarity transform used for point normalization."""
    points = np.asarray(points, dtype=float)

    centroid = np.mean(points, axis=0)
    shifted = points - centroid

    distances = np.linalg.norm(shifted, axis=1)
    mean_distance = np.mean(distances)

    if mean_distance < 1e-12:
        raise ValueError(
            "Point normalization is undefined for coincident points."
        )

    scale = np.sqrt(2.0) / mean_distance

    return np.array(
        [
            [scale, 0.0, -scale * centroid[0]],
            [0.0, scale, -scale * centroid[1]],
            [0.0, 0.0, 1.0],
        ],
        dtype=float,
    )

for view in views:
    T_image = normalize_trans(view["im_pts"])
    T_plane = normalize_trans(view["plane_pts"])

    view["T_image"] = T_image
    view["T_plane"] = T_plane
    view["normalized_image"] = (
        T_image @ homogenize(view["im_pts"]).T
    ).T
    view["normalized_plane"] = (
        T_plane @ homogenize(view["plane_pts"]).T
    ).T

print("Image and planar point normalization completed.")

## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

Each correspondence contributes two rows to $Q$. The last right singular vector gives the normalized homography parameters.

In [ ]:
def build_dlt_matrix(normalized_plane, normalized_image):
    """Build the 2N x 9 normalized DLT matrix Q."""
    n_points = normalized_image.shape[0]
    Q = np.zeros((2 * n_points, 9), dtype=float)

    for i in range(n_points):
        X, Y, _ = normalized_plane[i]
        u, v, _ = normalized_image[i]

        Q[2 * i] = [
            X, Y, 1.0,
            0.0, 0.0, 0.0,
            -u * X, -u * Y, -u,
        ]

        Q[2 * i + 1] = [
            0.0, 0.0, 0.0,
            X, Y, 1.0,
            -v * X, -v * Y, -v,
        ]

    return Q

for view in views:
    Q = build_dlt_matrix(
        view["normalized_plane"],
        view["normalized_image"],
    )

    _, singular_values_Q, Vt_Q = np.linalg.svd(Q)

    view["Q"] = Q
    view["dlt_singular_values"] = singular_values_Q
    view["H_normalized"] = Vt_Q[-1].reshape(3, 3)

print("Normalized DLT solved for all views.")

## 6. Denormalize Each Homography

The implementation computes

$$
H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}}
$$

and normalizes the result so that $H_{33}=1$.

In [ ]:
def denormalize_homography(H_normalized, T_image, T_plane):
    """Return the plane-to-image homography in original coordinates."""
    H = (
        np.linalg.inv(T_image)
        @ H_normalized
        @ T_plane
    )

    if abs(H[2, 2]) < 1e-12:
        raise ValueError(
            "Degenerate homography: H[2, 2] is too close to zero."
        )

    H /= H[2, 2]
    return H

expected_points = INTERNAL_CORNERS_X * INTERNAL_CORNERS_Y

for view in views:
    view["H"] = denormalize_homography(
        view["H_normalized"],
        view["T_image"],
        view["T_plane"],
    )

    if view["im_pts"].shape != (expected_points, 2):
        raise ValueError(
            f"Unexpected image-point shape for {view['name']}."
        )

    if view["plane_pts"].shape != (expected_points, 2):
        raise ValueError(
            f"Unexpected planar-point shape for {view['name']}."
        )

    if not np.all(np.isfinite(view["H"])):
        raise ValueError(
            f"Non-finite homography for {view['name']}."
        )

print("All homographies denormalized and validated.")

## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

Each valid homography contributes two Zhang constraints. All constraints are stacked into $V$ and solved using the last right singular vector.

In [ ]:
def v_ij(hi, hj):
    """Construct one six-element Zhang constraint vector."""
    return np.array(
        [
            hi[0] * hj[0],
            hi[0] * hj[1] + hi[1] * hj[0],
            hi[1] * hj[1],
            hi[2] * hj[0] + hi[0] * hj[2],
            hi[2] * hj[1] + hi[1] * hj[2],
            hi[2] * hj[2],
        ],
        dtype=float,
    )

def zhang_constraints(H):
    """Return the two independent Zhang constraints for one homography."""
    h1 = H[:, 0]
    h2 = H[:, 1]

    v12 = v_ij(h1, h2)
    v11 = v_ij(h1, h1)
    v22 = v_ij(h2, h2)

    return np.vstack((v12, v11 - v22))

V = np.vstack(
    [zhang_constraints(view["H"]) for view in views]
)

_, singular_values, Vt = np.linalg.svd(V)
b = Vt[-1]

print("Constraint matrix shape:", V.shape)
print("\nSingular values:")
print(singular_values)
print("\nResidual norm ||Vb||:")
print(np.linalg.norm(V @ b))
print("\nEstimated b vector:")
print(b)

## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

Recover the five intrinsic parameters from $b$, handle the homogeneous sign ambiguity when required, and assemble the intrinsic matrix $K$.

In [ ]:
b11, b12, b22, b13, b23, b33 = b

denominator = b11 * b22 - b12**2

if abs(denominator) < 1e-12:
    raise ValueError("Degenerate intrinsic calibration constraints.")

# Recover the vertical principal-point coordinate.
v0 = (b12 * b13 - b11 * b23) / denominator

# Scale factor used by the closed-form intrinsic solution.
lambda_ = b33 - (
    b13**2
    + v0 * (b12 * b13 - b11 * b23)
) / b11

# The homogeneous vector b is defined up to sign.
# Reverse it if needed so that the square-root terms remain physically valid.
if lambda_ / b11 <= 0 or lambda_ * b11 / denominator <= 0:
    b = -b
    b11, b12, b22, b13, b23, b33 = b

    denominator = b11 * b22 - b12**2
    v0 = (b12 * b13 - b11 * b23) / denominator

    lambda_ = b33 - (
        b13**2
        + v0 * (b12 * b13 - b11 * b23)
    ) / b11

alpha = np.sqrt(lambda_ / b11)
beta = np.sqrt(lambda_ * b11 / denominator)

gamma = -b12 * alpha**2 * beta / lambda_
u0 = gamma * v0 / beta - b13 * alpha**2 / lambda_

K = np.array(
    [
        [alpha, gamma, u0],
        [0.0, beta, v0],
        [0.0, 0.0, 1.0],
    ],
    dtype=float,
)

print("Intrinsic parameters")
print("--------------------")
print(f"alpha : {alpha:.6f}")
print(f"beta  : {beta:.6f}")
print(f"gamma : {gamma:.6f}")
print(f"u0    : {u0:.6f}")
print(f"v0    : {v0:.6f}")

print("\nIntrinsic matrix K:")
print(K)

## 9. Recover $R$ and $t$ for Every Retained View

Use `recover_extrinsic(K, H)` to recover the pose associated with each valid homography and project the approximate rotation onto the nearest proper rotation matrix.

In [ ]:
def recover_extrinsic(K, H):
    """Recover a proper rotation R and translation t from K and H."""
    K_inv = np.linalg.inv(K)

    h1 = H[:, 0]
    h2 = H[:, 1]
    h3 = H[:, 2]

    lambda_pose = 1.0 / np.linalg.norm(K_inv @ h1)

    r1 = lambda_pose * (K_inv @ h1)
    r2 = lambda_pose * (K_inv @ h2)
    r3 = np.cross(r1, r2)

    R_approx = np.column_stack((r1, r2, r3))

    U, _, Vt = np.linalg.svd(R_approx)
    R = U @ Vt

    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    t = lambda_pose * (K_inv @ h3)

    return R, t

for view in views:
    R, t = recover_extrinsic(K, view["H"])

    view["R"] = R
    view["t"] = t

    print(view["name"])
    print("R =")
    print(R)
    print("t =", t)
    print("-" * 60)

## 10. Reproject the $Z=0$ Calibration Points

Transform the planar points into the camera frame, apply $K$, and convert homogeneous image coordinates back to Cartesian pixel coordinates.

In [ ]:
for view in views:
    world_xy = view["plane_pts"]

    world_xyz = np.column_stack(
        (
            world_xy,
            np.zeros(world_xy.shape[0]),
        )
    )

    camera_points = (
        world_xyz @ view["R"].T
        + view["t"]
    )

    projected_h = camera_points @ K.T

    if np.any(np.abs(projected_h[:, 2]) < 1e-12):
        raise ValueError(
            f"Invalid homogeneous depth during reprojection for {view['name']}."
        )

    projected_pixels = (
        projected_h[:, :2]
        / projected_h[:, 2:3]
    )

    view["world_xyz"] = world_xyz
    view["projected"] = projected_pixels

print(f"Reprojected calibration views: {len(views)}")

## 11. Compute Point-wise Errors, Mean Error and RMSE

Evaluate image-space geometric consistency for every calibration point, each view, and the complete dataset.

In [ ]:
for view in views:
    residuals = (
        view["projected"]
        - view["im_pts"]
    )

    errors = np.linalg.norm(
        residuals,
        axis=1,
    )

    view["residuals"] = residuals
    view["errors"] = errors
    view["mean_error"] = float(errors.mean())
    view["rmse"] = float(
        np.sqrt(np.mean(errors**2))
    )

    print(view["name"])
    print(
        f"Mean reprojection error : "
        f"{view['mean_error']:.4f} px"
    )
    print(
        f"Reprojection RMSE       : "
        f"{view['rmse']:.4f} px"
    )
    print("-" * 60)

In [ ]:
all_errors = np.concatenate(
    [view["errors"] for view in views]
)

overall_mean_error = float(
    all_errors.mean()
)

overall_rmse = float(
    np.sqrt(np.mean(all_errors**2))
)

print("Calibration summary")
print("-------------------")
print(f"Valid calibration views : {len(views)}")
print(f"Calibration points       : {len(all_errors)}")
print(
    f"Mean reprojection error  : "
    f"{overall_mean_error:.4f} px"
)
print(
    f"Overall RMSE             : "
    f"{overall_rmse:.4f} px"
)

## 12. Produce and Save the Six Required Diagnostic Figures

All figures are written to `../outputs/figures/`.

In [ ]:
REQUIRED_OUTPUTS = [
    "detected_chessboard_corners.png",
    "homography_estimation_pipeline.png",
    "estimated_camera_poses.png",
    "reprojection_results.png",
    "mean_reprojection_error_by_view.png",
    "reprojection_error_distribution.png",
]

print("Diagnostic figures to generate:")
for output_name in REQUIRED_OUTPUTS:
    print(f"  - {output_name}")

### 12.1 Homography Estimation Pipeline

In [ ]:
# Summarize the normalized DLT homography estimation pipeline.
fig, ax = plt.subplots(figsize=(15, 3.5))
ax.axis("off")

labels = [
    "World points\n(X, Y, 1)",
    "Image points\n(u, v, 1)",
    "Normalize\nT_plane, T_image",
    "Build Q\nQh = 0",
    "SVD\nSolve for h",
    "Denormalize\nH",
]

x_positions = np.linspace(0.08, 0.92, len(labels))

for x, label in zip(x_positions, labels):
    ax.text(
        x,
        0.5,
        label,
        ha="center",
        va="center",
        fontsize=12,
        transform=ax.transAxes,
        bbox={
            "boxstyle": "round,pad=0.5",
            "fill": False,
            "linewidth": 1.5,
        },
    )

for start, end in zip(x_positions[:-1], x_positions[1:]):
    ax.annotate(
        "",
        xy=(end - 0.055, 0.5),
        xytext=(start + 0.055, 0.5),
        xycoords=ax.transAxes,
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 1.5,
        },
    )

ax.set_title("Homography Estimation Pipeline", fontsize=16, pad=20)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "homography_estimation_pipeline.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.2 Mean Reprojection Error by View

In [ ]:
image_names = [
    view["name"]
    for view in views
]

mean_errors = [
    view["mean_error"]
    for view in views
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    image_names,
    mean_errors,
)

ax.axhline(
    overall_mean_error,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall mean = {overall_mean_error:.3f} px",
)

for bar, value in zip(bars, mean_errors):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title("Mean Reprojection Error by Calibration View")
ax.set_xlabel("Calibration image")
ax.set_ylabel("Mean reprojection error (pixels)")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "mean_reprojection_error_by_view.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.3 Reprojection Error Distribution

In [ ]:
# Analyse the distribution of point-wise reprojection errors.
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    all_errors,
    bins=20,
    edgecolor="black",
    alpha=0.8,
)

ax.axvline(
    overall_mean_error,
    linestyle="--",
    linewidth=1.5,
    label=f"Mean = {overall_mean_error:.3f} px",
)

ax.axvline(
    overall_rmse,
    linestyle=":",
    linewidth=1.5,
    label=f"RMSE = {overall_rmse:.3f} px",
)

ax.set_title("Distribution of Reprojection Errors")
ax.set_xlabel("Reprojection error (pixels)")
ax.set_ylabel("Number of calibration points")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "reprojection_error_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.4 Detected Chessboard Corners

In [ ]:
n_images = len(views)
n_cols = 3
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 5 * n_rows),
)

axes = np.asarray(axes).reshape(-1)

for ax, view in zip(axes, views):
    visual = view["image"].copy()

    corners = (
        view["im_pts"]
        .reshape(-1, 1, 2)
        .astype(np.float32)
    )

    cv2.drawChessboardCorners(
        visual,
        (
            INTERNAL_CORNERS_X,
            INTERNAL_CORNERS_Y,
        ),
        corners,
        True,
    )

    visual_rgb = cv2.cvtColor(
        visual,
        cv2.COLOR_BGR2RGB,
    )

    ax.imshow(visual_rgb)
    ax.set_title(view["name"])
    ax.axis("off")

for ax in axes[n_images:]:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "detected_chessboard_corners.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.5 Estimated Camera Poses

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

reference_xy = views[0]["plane_pts"]
reference_xyz = np.column_stack(
    (
        reference_xy,
        np.zeros(reference_xy.shape[0]),
    )
)

ax.scatter(
    reference_xyz[:, 0],
    reference_xyz[:, 1],
    reference_xyz[:, 2],
    marker="s",
    label="Chessboard",
)

for view in views:
    R = view["R"]
    t = view["t"]

    camera_center = -R.T @ t

    ax.scatter(
        camera_center[0],
        camera_center[1],
        camera_center[2],
        marker="o",
    )

    ax.text(
        camera_center[0],
        camera_center[1],
        camera_center[2],
        view["path"].stem,
        fontsize=8,
    )

ax.set_title("Estimated Camera Poses")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "estimated_camera_poses.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### 12.6 Detected vs Reprojected Points

In [ ]:
n_images = len(views)
n_cols = 3
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 5 * n_rows),
)

axes = np.asarray(axes).reshape(-1)

for ax, view in zip(axes, views):
    image_rgb = cv2.cvtColor(
        view["image"],
        cv2.COLOR_BGR2RGB,
    )

    ax.imshow(image_rgb)

    ax.scatter(
        view["im_pts"][:, 0],
        view["im_pts"][:, 1],
        s=24,
        marker="o",
        label="Detected",
    )

    ax.scatter(
        view["projected"][:, 0],
        view["projected"][:, 1],
        s=24,
        marker="x",
        label="Reprojected",
    )

    ax.set_title(
        f"{view['name']} — RMSE: {view['rmse']:.3f} px"
    )
    ax.axis("off")
    ax.legend(fontsize=8)

for ax in axes[n_images:]:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "reprojection_results.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 13. Run the Numerical and Output-file Validation Checks

Validate $K$, rotation orthonormality and determinant, finite residuals, and the existence of all six required output figures.

In [ ]:
if K.shape != (3, 3) or not np.all(np.isfinite(K)):
    raise ValueError("Invalid intrinsic matrix K.")

for view in views:
    if not np.all(np.isfinite(view["H"])):
        raise ValueError(
            f"Non-finite homography for {view['name']}."
        )

    R = view["R"]

    if not np.allclose(
        R.T @ R,
        np.eye(3),
        atol=1e-6,
    ):
        raise ValueError(
            f"Non-orthonormal rotation for {view['name']}."
        )

    if not np.isclose(
        np.linalg.det(R),
        1.0,
        atol=1e-6,
    ):
        raise ValueError(
            f"Invalid rotation determinant for {view['name']}."
        )

    if not np.all(np.isfinite(view["errors"])):
        raise ValueError(
            f"Non-finite residuals for {view['name']}."
        )

missing = [
    name
    for name in REQUIRED_OUTPUTS
    if not (OUTPUT_DIR / name).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing outputs: " + ", ".join(missing)
    )

print("All Camera Calibration validation checks passed.")

## Final Result Summary

The pipeline estimates camera intrinsics from multiple planar chessboard views, recovers one pose per valid image, and evaluates geometric consistency through reprojection. Lens-distortion estimation is intentionally outside the scope of this implementation.